# Serving — agg_video_daily / agg_channel_daily (last read of each day)

Added 2026-11-02. The Gold facts keep **every read** (`fact_video_snapshot`, `fact_channel_snapshot`, grain id × `snapshot_ts`). Dashboards, the app, `agg_restaurant_daily_views` and `agg_restaurant_ranking` work per day, so this notebook builds the per-day serving tables (same columns as the old `fact_video_daily_snapshot` / `fact_channel_daily_snapshot`, plus `reads_in_day`). Other grains (per hour, ...) can be added here when a dashboard needs them.

| Table | Grain | Value |
| --- | --- | --- |
| `agg_video_daily` | video × Vietnam day | counts of the **last read of the day**; `*_gained` vs the previous day row (additive over days, NULL on the first day); `measured_at` = time of that read |
| `agg_channel_daily` | channel × Vietnam day | same |

**Load:** `replaceWhere snapshot_date_key >= D` (D = last loaded day, so a rerun or an intraday refresh replaces today's row). Runs after `gold/facts` in both jobs (daily 01h and stats refresh 07 / 09 / 21h).

In [ ]:
for layer in ("bronze", "silver", "gold"):
    dbutils.widgets.text(layer, f"prod.{layer}")
BRONZE, SILVER, GOLD = (dbutils.widgets.get(layer) for layer in ("bronze", "silver", "gold"))


def reload_from_key(table):
    """First snapshot_date_key to rebuild: the last loaded day (a rerun refreshes it) and every newer day."""
    if not spark.catalog.tableExists(table):
        return 0
    return spark.sql(f"SELECT MAX(snapshot_date_key) AS k FROM {table}").collect()[0]["k"] or 0


def build_daily(fact, target, key, counts, gains, extra_cols, ddl, comment):
    """Last read of each (key, day) of `fact` -> `target`; gains vs the previous day row."""
    spark.sql(f"CREATE TABLE IF NOT EXISTS {target} ({ddl}) USING DELTA CLUSTER BY (snapshot_date_key) COMMENT '{comment}'")
    from_key = reload_from_key(target)
    struct_cols = ", ".join(f"'{c}', {c}" for c in counts + extra_cols + ["snapshot_ts"])
    picked = ", ".join(f"r.{c} AS {c}" for c in counts + extra_cols)
    gain_cols = ", ".join(f"d.{c} - LAG(d.{c}) OVER w AS {g}" for c, g in zip(counts, gains))
    df = spark.sql(f"""
        WITH day_last AS (
            SELECT snapshot_date_key, {key},
                   max_by(named_struct({struct_cols}), snapshot_ts) AS r,
                   CAST(COUNT(*) AS INT) AS reads_in_day
            FROM {fact}
            GROUP BY snapshot_date_key, {key}
        ),
        d AS (SELECT snapshot_date_key, {key}, {picked}, r.snapshot_ts AS measured_at, reads_in_day FROM day_last)
        SELECT d.snapshot_date_key, d.{key}, {", ".join("d." + c for c in counts + extra_cols)},
               {gain_cols},
               LAG(d.snapshot_date_key) OVER w IS NULL AS is_first_snapshot,
               d.reads_in_day, d.measured_at, current_timestamp() AS etl_date
        FROM d
        WINDOW w AS (PARTITION BY d.{key} ORDER BY d.snapshot_date_key)
    """).where(f"snapshot_date_key >= {from_key}")
    df = df.select(*spark.table(target).columns)
    n = df.count()
    df.write.format("delta").mode("overwrite").option("replaceWhere", f"snapshot_date_key >= {from_key}").saveAsTable(target)
    print(f"✅ {target}: replaced snapshot_date_key >= {from_key} with {n} rows")

In [ ]:
build_daily(
    fact=f"{GOLD}.fact_video_snapshot", target=f"{GOLD}.agg_video_daily", key="video_key",
    counts=["view_count", "like_count", "comment_count"],
    gains=["views_gained", "likes_gained", "comments_gained"],
    extra_cols=["channel_key", "video_age_days"],
    ddl="""snapshot_date_key INT, video_key BIGINT, channel_key BIGINT, view_count BIGINT, like_count BIGINT,
           comment_count BIGINT, views_gained BIGINT, likes_gained BIGINT, comments_gained BIGINT,
           video_age_days INT, is_first_snapshot BOOLEAN, reads_in_day INT, measured_at TIMESTAMP, etl_date TIMESTAMP""",
    comment="Serving: 1 video x 1 Vietnam day, last read of the day (from fact_video_snapshot). Counts semi-additive, gains vs previous day additive.",
)

In [ ]:
build_daily(
    fact=f"{GOLD}.fact_channel_snapshot", target=f"{GOLD}.agg_channel_daily", key="channel_key",
    counts=["subscriber_count", "view_count", "video_count"],
    gains=["subscribers_gained", "views_gained", "videos_gained"],
    extra_cols=[],
    ddl="""snapshot_date_key INT, channel_key BIGINT, subscriber_count BIGINT, view_count BIGINT, video_count INT,
           subscribers_gained BIGINT, views_gained BIGINT, videos_gained INT, is_first_snapshot BOOLEAN,
           reads_in_day INT, measured_at TIMESTAMP, etl_date TIMESTAMP""",
    comment="Serving: 1 channel x 1 Vietnam day, last read of the day (from fact_channel_snapshot). Counts semi-additive, gains vs previous day additive.",
)

In [ ]:
for t, key in [("agg_video_daily", "video_key"), ("agg_channel_daily", "channel_key")]:
    s = spark.sql(f"""SELECT COUNT(*) AS n, COUNT(DISTINCT snapshot_date_key, {key}) AS grain, MAX(snapshot_date_key) AS last_day,
                             ROUND(AVG(reads_in_day), 2) AS avg_reads FROM {GOLD}.{t}""").collect()[0]
    print(f"📊 {GOLD}.{t}: {s['n']} rows · last day {s['last_day']} · {s['avg_reads']} reads per row on average · grain: "
          + ("✅" if s["n"] == s["grain"] else "❌ duplicates"))
print("\n--- agg_daily_snapshots complete ---")